In [ ]:
pip install -U giotto-tda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.4/526.4 kB 28.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.9/452.9 kB 32.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 55.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.0 MB/s eta 0:00:00


In [ ]:
import numpy as np

from gtda.homology import VietorisRipsPersistence
from gtda.time_series import SingleTakensEmbedding
from gtda.diagrams import BettiCurve, PersistenceLandscape, PairwiseDistance, PersistenceEntropy
from gtda.plotting import plot_point_cloud, plot_diagram
from gtda.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from pathlib import Path
from sklearn.datasets import make_circles
import plotly.graph_objects as go

In [ ]:
np.random.seed(seed=42)

X = np.asarray([
    make_circles(100, factor=np.random.random())[0]
    for i in range(10)
])

In [ ]:
i = 4
plot_point_cloud(X[i])

In [ ]:
VR = VietorisRipsPersistence()
Xt = VR.fit_transform(X)
VR.plot(Xt, sample=i)

In [ ]:
BC = BettiCurve()
X_betti_curve = BC.fit_transform_plot(Xt)

In [ ]:
PL = PersistenceLandscape()
X_persistence_landscape = PL.fit_transform_plot(Xt)

In [ ]:
def basic_time_series(n, b, y):
    r = np.random.rand(n)
    j = 2 * np.pi + np.arcsin(r)
    t = np.arange(n)
    x = b * np.sin(y * t + j)
    return x


def noisy_time_series(n, b, y):
    r = np.random.rand(n)
    j = 2 * np.pi * r
    t = np.arange(n)
    x = b * np.sin(y * t + j)
    return x

In [ ]:
n = 500
b = 1.5
y = 1

basic_ts = basic_time_series(n, b, y)
noisy_ts = noisy_time_series(n, b, y)

In [ ]:
fig = go.Figure(data=go.Scatter(y=basic_ts))
fig.update_layout(xaxis_title="Timestamp", yaxis_title="Amplitude")
fig.show()

In [ ]:
fig = go.Figure(data=go.Scatter(y=noisy_ts))
fig.update_layout(xaxis_title="Timestamp", yaxis_title="Amplitude")
fig.show()

In [ ]:
embedding_dimension_periodic = 3
embedding_time_delay_periodic = 8

embedder = SingleTakensEmbedding(
    parameters_type="fixed",
    time_delay=embedding_time_delay_periodic,
    dimension=embedding_dimension_periodic
)

basic_point_cloud = embedder.fit_transform(basic_ts)
noisy_point_cloud = embedder.fit_transform(noisy_ts)

In [ ]:
plot_point_cloud(basic_point_cloud).show()

In [ ]:
plot_point_cloud(noisy_point_cloud).show()

In [ ]:
# 0 - connected components, 1 - loops, 2 - voids
homology_dimensions = [0, 1, 2]

persistence = VietorisRipsPersistence(
    homology_dimensions=homology_dimensions
)

In [ ]:
diagrams = persistence.fit_transform([basic_point_cloud, noisy_point_cloud])

In [ ]:
VR.plot(diagrams, sample=0)

In [ ]:
VR.plot(diagrams, sample=1)

In [ ]:
p_W = 2
PD = PairwiseDistance(metric='wasserstein',
                      metric_params={'p': p_W, 'delta': 0.1},
                      order=None)

PD.fit_transform(diagrams)

array([[[0.        , 0.        , 0.        ],
        [1.29031027, 0.9218511 , 0.2346551 ]],

       [[1.289664  , 0.92093727, 0.23481503],
        [0.        , 0.        , 0.        ]]])

In [ ]:
PD = PairwiseDistance(metric='bottleneck',
                      metric_params={'delta': 0.1},
                      order=None)

PD.fit_transform(diagrams)

array([[[0.        , 0.        , 0.        ],
        [0.07992657, 0.63924557, 0.14452577]],

       [[0.07800749, 0.63128385, 0.14573882],
        [0.        , 0.        , 0.        ]]])

In [ ]:
BC = BettiCurve()
betti_curves = BC.fit_transform(diagrams)

In [ ]:
BC.plot(betti_curves, sample=0)

In [ ]:
BC.plot(betti_curves, sample=1)

In [ ]:
PD = PairwiseDistance(metric='betti',
                      metric_params={'p': 1},
                      order=None)

PD.fit_transform(diagrams)

array([[[ 0.        ,  0.        ,  0.        ],
        [27.95031904, 11.09281855,  1.68033309]],

       [[27.95031904, 11.09281855,  1.68033309],
        [ 0.        ,  0.        ,  0.        ]]])

In [ ]:
PL = PersistenceLandscape()
persistence_landscapes = PL.fit_transform(diagrams)

In [ ]:
PL.plot(persistence_landscapes, sample=0)

In [ ]:
PL.plot(persistence_landscapes, sample=1)

In [ ]:
PD = PairwiseDistance(metric='landscape',
                      metric_params={'p': 1},
                      order=None)

PD.fit_transform(diagrams)

array([[[0.        , 0.        , 0.        ],
        [0.00949094, 0.29436653, 0.02973432]],

       [[0.00949094, 0.29436653, 0.02973432],
        [0.        , 0.        , 0.        ]]])

In [ ]:
def make_point_clouds(n_samples_per_shape: int, n_points: int, noise: float):
    """Make point clouds for circles, spheres, and tori with random noise.
    """
    circle_point_clouds = [
        np.asarray(
            [
                [np.sin(t) + noise * (np.random.rand(1)[0] - 0.5), np.cos(t) + noise * (np.random.rand(1)[0] - 0.5), 0]
                for t in range((n_points ** 2))
            ]
        )
        for kk in range(n_samples_per_shape)
    ]
    # label circles with 0
    circle_labels = np.zeros(n_samples_per_shape)

    sphere_point_clouds = [
        np.asarray(
            [
                [
                    np.cos(s) * np.cos(t) + noise * (np.random.rand(1)[0] - 0.5),
                    np.cos(s) * np.sin(t) + noise * (np.random.rand(1)[0] - 0.5),
                    np.sin(s) + noise * (np.random.rand(1)[0] - 0.5),
                ]
                for t in range(n_points)
                for s in range(n_points)
            ]
        )
        for kk in range(n_samples_per_shape)
    ]
    # label spheres with 1
    sphere_labels = np.ones(n_samples_per_shape)

    torus_point_clouds = [
        np.asarray(
            [
                [
                    (2 + np.cos(s)) * np.cos(t) + noise * (np.random.rand(1)[0] - 0.5),
                    (2 + np.cos(s)) * np.sin(t) + noise * (np.random.rand(1)[0] - 0.5),
                    np.sin(s) + noise * (np.random.rand(1)[0] - 0.5),
                ]
                for t in range(n_points)
                for s in range(n_points)
            ]
        )
        for kk in range(n_samples_per_shape)
    ]
    # label tori with 2
    torus_labels = 2 * np.ones(n_samples_per_shape)

    point_clouds = np.concatenate((circle_point_clouds, sphere_point_clouds, torus_point_clouds))
    labels = np.concatenate((circle_labels, sphere_labels, torus_labels))

    return point_clouds, labels

In [ ]:
point_clouds_basic, labels_basic = make_point_clouds(n_samples_per_shape=10, n_points=20, noise=0.5)
point_clouds_basic.shape, labels_basic.shape

((30, 400, 3), (30,))

In [ ]:
plot_point_cloud(point_clouds_basic[0])

In [ ]:
plot_point_cloud(point_clouds_basic[10])

In [ ]:
plot_point_cloud(point_clouds_basic[-1])

In [ ]:
# Track connected components, loops, and voids
homology_dimensions = [0, 1, 2]

# Collapse edges to speed up H2 persistence calculation!
persistence = VietorisRipsPersistence(
    metric="euclidean",
    homology_dimensions=homology_dimensions,
    collapse_edges=True,
)

diagrams_basic = persistence.fit_transform(point_clouds_basic)

In [ ]:
# Circle
plot_diagram(diagrams_basic[0])

In [ ]:
#spheres
plot_diagram(diagrams_basic[10])

In [ ]:
#tori
plot_diagram(diagrams_basic[-1])

In [ ]:
persistence_entropy = PersistenceEntropy()

# calculate topological feature matrix
X_basic = persistence_entropy.fit_transform(diagrams_basic)

In [ ]:
plot_point_cloud(X_basic)

In [ ]:
rf = RandomForestClassifier(oob_score=True)
rf.fit(X_basic, labels_basic)

print(f"OOB score: {rf.oob_score_:.3f}")

OOB score: 1.000


In [ ]:
steps = [
    ("persistence", VietorisRipsPersistence(metric="euclidean", homology_dimensions=homology_dimensions, n_jobs=6)),
    ("entropy", PersistenceEntropy()),
    ("model", RandomForestClassifier(oob_score=True)),
]

pipeline = Pipeline(steps)

pipeline.fit(point_clouds_basic, labels_basic)

Pipeline(steps=[('persistence',
                 VietorisRipsPersistence(homology_dimensions=[0, 1, 2])),
                ('entropy', PersistenceEntropy()),
                ('model', RandomForestClassifier(oob_score=True))])

pipeline["model"].oob_score_

1.0